# Road Surface Condition Classification: Stage 0 and Stage 1
**Stage 0:** Data understanding  |  **Stage 1:** Data cleaning and preprocessing (Computer Vision)

Textbook mapping: this notebook is the *Data Sources -> Transformation (select, extract, cleanse, integrate, load) -> Data Mart* part of the DW architecture. The output is a clean, split, resized dataset that Stage 2 (CNN) will read.

**Before you run:** download the dataset from Mendeley (https://data.mendeley.com/datasets/c43y93xswd/1), put the zip in your Google Drive at `MyDrive/road_project/dataset.zip`, then run the cells top to bottom. Cells marked **CHECK** need a human decision.

## 0. Setup

In [ ]:
!pip -q install imagehash

import os, json, zipfile, hashlib, shutil, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import imagehash
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/road_project')
ZIP_PATH   = DRIVE_ROOT / 'dataset.zip'     # <- the Mendeley zip you uploaded
RAW_DIR    = Path('/content/raw')           # extracted here (fast local disk)
WORK_DIR   = Path('/content/work')          # processed output
OUT_DIR    = DRIVE_ROOT / 'outputs'         # reports and splits saved back to Drive

# ---- Cleaning / preprocessing settings ----
IMG_SIZE         = 224          # matches MobileNetV2 / EfficientNetB0 default input
PAD_TO_SQUARE    = False        # False = direct resize, True = letterbox (keeps aspect ratio)
PHASH_THRESHOLD  = 4            # Hamming distance (0-64). <=4 treats images as near-duplicates
SPLIT            = (0.70, 0.15, 0.15)   # train / val / test
DROP_LOW_QUALITY = False        # flags are always shown; set True to also drop them

# Expected classes from the project objectives
EXPECTED = ['crack', 'pothole', 'damaged asphalt', 'water-filled pothole', 'open manhole', 'normal']

for d in (RAW_DIR, WORK_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## Stage 0: Data understanding

In [ ]:
# Extract the zip (and any zips nested inside it)
assert ZIP_PATH.exists(), f"Put the dataset zip at {ZIP_PATH}"
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(RAW_DIR)
for nz in list(RAW_DIR.rglob('*.zip')):
    with zipfile.ZipFile(nz) as z:
        z.extractall(nz.with_suffix(''))

# Print the folder structure so we can see how the dataset is organized
def tree(root, max_depth=3):
    root = Path(root)
    for p in sorted(root.rglob('*')):
        depth = len(p.relative_to(root).parts)
        if p.is_dir() and depth <= max_depth:
            n = sum(1 for f in p.iterdir() if f.is_file())
            print('  ' * (depth - 1) + f'{p.name}/  ({n} files)')
tree(RAW_DIR)

**CHECK:** Read the tree above. This notebook assumes each image sits in a folder named after its class. If the dataset instead uses one folder plus a CSV or annotation file for labels, edit `get_label()` in the next cell.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def get_label(p: Path) -> str:
    return p.parent.name          # <- edit if labels come from somewhere else

rows = []
for p in RAW_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in IMG_EXT and '__MACOSX' not in p.parts:
        rows.append({'path': str(p), 'label': get_label(p), 'ext': p.suffix.lower(),
                     'kb': p.stat().st_size / 1024})
df = pd.DataFrame(rows)
print(f'{len(df)} image files found')
df['label'].value_counts()

In [ ]:
# CHECK: compare the folder names to the six classes in the objectives.
# Fill LABEL_MAP to rename or merge, e.g. {'Pothole': 'pothole', 'Potholes': 'pothole'}
LABEL_MAP = {}
df['label'] = df['label'].replace(LABEL_MAP)

found = sorted(df['label'].unique())
print('Found  :', found)
print('Expected:', EXPECTED)
print('\nIn dataset but not in objectives:', sorted(set(found) - set(EXPECTED)))
print('In objectives but not in dataset:', sorted(set(EXPECTED) - set(found)))

In [ ]:
# Class distribution
counts = df['label'].value_counts()
ax = counts.plot(kind='bar', figsize=(8, 4), color='steelblue')
ax.set_title('Images per class (raw)'); ax.set_ylabel('count')
for i, v in enumerate(counts.values): ax.text(i, v, str(v), ha='center', va='bottom')
plt.tight_layout(); plt.show()
print(f'Imbalance ratio (largest / smallest class): {counts.max() / counts.min():.1f}x')

In [ ]:
# Basic image properties (header read only, so this is fast) + corrupt check
def probe(path):
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            return im.size[0], im.size[1], im.mode, True
    except Exception:
        return None, None, None, False

with ThreadPoolExecutor(8) as ex:
    res = list(tqdm(ex.map(probe, df['path']), total=len(df)))
df[['w', 'h', 'mode', 'readable']] = pd.DataFrame(res, index=df.index)

print('Unreadable files:', (~df['readable']).sum())
print('\nColor modes:\n', df['mode'].value_counts(dropna=False))
print('\nImage size summary:')
print(df[['w', 'h', 'kb']].describe().round(1))
print('\nMost common resolutions:')
print(df.groupby(['w', 'h']).size().sort_values(ascending=False).head(10))

In [ ]:
# Look at the data: random samples per class. Look for mislabeled or odd images.
def show_grid(frame, per_class=5, seed=SEED):
    labels = sorted(frame['label'].unique())
    fig, axes = plt.subplots(len(labels), per_class, figsize=(3 * per_class, 2.6 * len(labels)))
    axes = np.atleast_2d(axes)
    for r, lab in enumerate(labels):
        sub = frame[frame['label'] == lab]
        pick = sub.sample(min(per_class, len(sub)), random_state=seed)
        for c in range(per_class):
            ax = axes[r, c]; ax.axis('off')
            if c < len(pick):
                img = cv2.imread(pick.iloc[c]['path'])
                if img is not None:
                    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if c == 0: ax.set_title(lab, loc='left', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

show_grid(df[df['readable']])

**Stage 0 checklist for the team** (write the answers in your paper's "Data Description" section):
- Do the class names and counts match the objectives? Which classes are missing or merged?
- Are images all the same size, or mixed? Any grayscale or PNG-with-alpha files?
- Which classes look visually confusable (crack vs damaged asphalt, pothole vs water-filled pothole)?
- Did you spot mislabeled samples in the grids? Re-run `show_grid` with different seeds.
- Is the dataset already split into train/test? If so, we still re-split after cleaning, so duplicates cannot leak across sets.

## Stage 1: Clean the dataset (Computer Vision)
One pass over every image computes: full-decode check (OpenCV), MD5 (exact duplicates), perceptual hash (near duplicates), blur score (variance of the Laplacian), and mean brightness.

In [ ]:
def analyze(path):
    out = {'path': path, 'ok': False}
    try:
        data = Path(path).read_bytes()
        out['md5'] = hashlib.md5(data).hexdigest()
        arr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        if arr is None or arr.size == 0:
            return out
        gray = cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
        out['blur'] = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        out['brightness'] = float(gray.mean())
        pil = Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))
        out['phash'] = int(str(imagehash.phash(pil)), 16)
        out['ok'] = True
    except Exception as e:
        out['err'] = str(e)
    return out

with ThreadPoolExecutor(8) as ex:
    stats = list(tqdm(ex.map(analyze, df['path']), total=len(df)))
stats = pd.DataFrame(stats)
work = df.merge(stats, on='path')
print('Failed full decode:', (~work['ok']).sum())

In [ ]:
# 1) Remove corrupted / undecodable images
corrupt = work[~work['ok']]
corrupt[['path', 'label']].to_csv(OUT_DIR / 'removed_corrupt.csv', index=False)
work = work[work['ok']].copy().reset_index(drop=True)
print(f'Removed {len(corrupt)} corrupt images, {len(work)} remain')

In [ ]:
# 2) Exact + near-duplicate detection (perceptual hash, clustered with union-find)
def popcount64(x):
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return (x * np.uint64(0x0101010101010101)) >> np.uint64(56)

H = np.array([np.uint64(v) for v in work['phash']], dtype=np.uint64)
n = len(H)
parent = list(range(n))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

for i in tqdm(range(n - 1)):
    d = popcount64(H[i] ^ H[i + 1:])
    for j in np.nonzero(d <= PHASH_THRESHOLD)[0]:
        union(i, i + 1 + int(j))

work['cluster'] = [find(i) for i in range(n)]
sizes = work['cluster'].map(work['cluster'].value_counts())
dups = work[sizes > 1]
n_exact = work.duplicated('md5').sum()
print(f'Exact duplicates (same bytes): {n_exact}')
print(f'Images in near-duplicate clusters: {len(dups)} across {dups["cluster"].nunique()} clusters')

In [ ]:
# Handle clusters:
#  - same label      -> keep the largest file, drop the rest
#  - conflicting labels (same picture labeled as 2+ classes) -> drop the whole cluster
keep_idx, drop_rows, conflict_clusters = [], [], []
for cid, g in work.groupby('cluster'):
    if len(g) == 1:
        keep_idx.append(g.index[0]); continue
    if g['label'].nunique() > 1:
        conflict_clusters.append(cid)
        drop_rows += [(p, l, 'label_conflict') for p, l in zip(g['path'], g['label'])]
    else:
        best = g['kb'].idxmax()
        keep_idx.append(best)
        drop_rows += [(p, l, 'duplicate') for i, p, l in zip(g.index, g['path'], g['label']) if i != best]

pd.DataFrame(drop_rows, columns=['path', 'label', 'reason']).to_csv(OUT_DIR / 'removed_duplicates.csv', index=False)
print(f'Duplicate removal: kept {len(keep_idx)}, dropped {len(drop_rows)} '
      f'({len(conflict_clusters)} clusters had conflicting labels)')
work = work.loc[keep_idx].reset_index(drop=True)

In [ ]:
# Peek at some detected duplicate pairs (sanity check that the threshold is sensible)
dropped = pd.read_csv(OUT_DIR / 'removed_duplicates.csv')
if len(dropped):
    sample = dropped.sample(min(6, len(dropped)), random_state=SEED)
    fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
    axes = np.atleast_1d(axes)
    for ax, (_, r) in zip(axes, sample.iterrows()):
        ax.imshow(cv2.cvtColor(cv2.imread(r['path']), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{r['label']}\n{r['reason']}", fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

**CHECK:** Do the dropped images look like true duplicates? If they are just similar-looking but different road scenes, lower `PHASH_THRESHOLD` (e.g. 2). If obvious duplicates were missed, raise it (e.g. 6) and re-run from the perceptual hash cell.

In [ ]:
# 3) Quality flags: blur (variance of Laplacian) and exposure. Flag, review, then decide.
blur_cut = work['blur'].quantile(0.02)
work['low_blur']  = work['blur'] <= blur_cut
work['too_dark']  = work['brightness'] < 30
work['too_light'] = work['brightness'] > 225
work['flagged']   = work[['low_blur', 'too_dark', 'too_light']].any(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(np.log1p(work['blur']), bins=50, ax=axes[0]); axes[0].set_title('log(1 + blur score)')
sns.histplot(work['brightness'], bins=50, ax=axes[1]); axes[1].set_title('mean brightness (0-255)')
plt.tight_layout(); plt.show()

print(work[['low_blur', 'too_dark', 'too_light']].sum().to_string())
print('\nFlagged per class:\n', work[work['flagged']]['label'].value_counts().to_string())
show_grid(work[work['flagged']], per_class=5)

In [ ]:
if DROP_LOW_QUALITY:
    work[work['flagged']][['path', 'label']].to_csv(OUT_DIR / 'removed_low_quality.csv', index=False)
    work = work[~work['flagged']].reset_index(drop=True)
print(f'{len(work)} images after cleaning')
print(work['label'].value_counts().to_string())

### Split first, augment later
The split happens **after** duplicate removal and **before** any augmentation. That way no copy of a test image can appear in training. Augmentation is applied on-the-fly to the training set only (see below).

In [ ]:
tr, va, te = SPLIT
assert abs(tr + va + te - 1) < 1e-9
train_df, temp_df = train_test_split(work, test_size=1 - tr, stratify=work['label'], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=te / (va + te), stratify=temp_df['label'], random_state=SEED)

train_df = train_df.assign(split='train'); val_df = val_df.assign(split='val'); test_df = test_df.assign(split='test')
splits = pd.concat([train_df, val_df, test_df])
splits[['path', 'label', 'split']].to_csv(OUT_DIR / 'splits.csv', index=False)

print(pd.crosstab(splits['label'], splits['split'], margins=True))

In [ ]:
# 4) Preprocess + export: standardize to RGB, resize to IMG_SIZE x IMG_SIZE, save as JPEG.
#    Pixel normalization (0-255 -> 0-1) is done inside the model with a Rescaling layer in Stage 2,
#    so the files stay as normal uint8 images.
def resize_img(img, size, pad):
    if not pad:
        return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size, size, 3), np.uint8)
    y0, x0 = (size - nh) // 2, (size - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = img
    return canvas

def safe(s): return str(s).strip().replace(' ', '_').replace('/', '_')

PROC = WORK_DIR / 'processed'
if PROC.exists(): shutil.rmtree(PROC)

def export_one(args):
    i, path, label, split = args
    img = cv2.imdecode(np.fromfile(path, np.uint8), cv2.IMREAD_COLOR)   # always 3-channel BGR
    img = resize_img(img, IMG_SIZE, PAD_TO_SQUARE)
    out = PROC / split / safe(label)
    out.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out / f'{i:06d}.jpg'), img, [cv2.IMWRITE_JPEG_QUALITY, 95])

jobs = [(i, r.path, r.label, r.split) for i, r in enumerate(splits.itertuples())]
with ThreadPoolExecutor(8) as ex:
    list(tqdm(ex.map(export_one, jobs), total=len(jobs)))

for s in ('train', 'val', 'test'):
    print(s, sum(1 for _ in (PROC / s).rglob('*.jpg')), 'images')

### Class imbalance and augmentation
Two tools, both applied to **training data only**:
1. **Class weights** are used in the loss so rare classes count more.
2. **On-the-fly augmentation** (flip, small rotation, zoom, brightness/contrast) gives the model varied views without saving copies. No vertical flip, since roads have a fixed perspective and upside-down road images are unrealistic.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

def load(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        PROC / split, image_size=(IMG_SIZE, IMG_SIZE), batch_size=32,
        label_mode='int', shuffle=shuffle, seed=SEED)

train_ds, val_ds, test_ds = load('train', True), load('val', False), load('test', False)
class_names = train_ds.class_names
print('Class order:', class_names)

# Class weights from the training split
y_train = np.array(train_df['label'].map(lambda s: class_names.index(safe(s))))
cw = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_train)
class_weights = {i: float(w) for i, w in enumerate(cw)}
print('Class weights:', {class_names[i]: round(w, 2) for i, w in class_weights.items()})

# Augmentation block: attach to the training pipeline ONLY (inside the model or via .map)
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
    layers.RandomContrast(0.2),
], name='augment')

# Preview: one training image, 8 random augmentations
img, _ = next(iter(train_ds.take(1)))
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for ax in axes:
    ax.imshow(np.clip(augment(img[:1], training=True)[0].numpy(), 0, 255).astype('uint8')); ax.axis('off')
plt.show()

In [ ]:
# Save the processed dataset (zipped) and the metadata to Drive so Stage 2 can start from here
archive = shutil.make_archive(str(OUT_DIR / 'processed_dataset'), 'zip', PROC)
json.dump({'class_names': class_names, 'class_weights': class_weights, 'img_size': IMG_SIZE,
           'split': SPLIT, 'seed': SEED, 'phash_threshold': PHASH_THRESHOLD,
           'pad_to_square': PAD_TO_SQUARE},
          open(OUT_DIR / 'preprocessing_config.json', 'w'), indent=2)

summary = {
    'raw_images': int(len(df)),
    'corrupt_removed': int(len(corrupt)),
    'duplicates_or_conflicts_removed': int(len(drop_rows)),
    'final_images': int(len(splits)),
}
json.dump(summary, open(OUT_DIR / 'cleaning_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))
print('Saved to:', OUT_DIR)